In [1]:
from google.colab import files
uploaded = files.upload()

import torch

def load_data(path="TrainingNames.txt"):
    with open(path, "r", encoding="utf-8") as f:
        names = f.read().splitlines()

    names = ["^" + name.lower().strip() + "$" for name in names]

    #  remove duplicates (IMPORTANT)
    names = list(set(names))

    return names

names = load_data()

print("Total names:", len(names))
print("Sample:", names[0])

Saving TrainingNames.txt to TrainingNames.txt
Total names: 895
Sample: ^ishaan kulkarni$


In [2]:
chars = sorted(list(set("".join(names))))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

print("Vocab size:", vocab_size)
print(chars)

Vocab size: 27
[' ', '$', '^', 'a', 'b', 'c', 'd', 'e', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [3]:
def encode(name):
    return torch.tensor([stoi[ch] for ch in name], dtype=torch.long)

X, Y = [], []

for name in names:
    encoded = encode(name)
    X.append(encoded[:-1])
    Y.append(encoded[1:])

print("Prepared dataset size:", len(X))

Prepared dataset size: 895


## RNN Model

In [4]:
import torch.nn as nn

class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        x = self.embedding(x)
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h

In [5]:
import torch.optim as optim

hidden_size = 128
lr = 0.001
epochs = 10

model = CharRNN(vocab_size, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):
    total_loss = 0

    for x, y in zip(X, Y):
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

        optimizer.zero_grad()
        output, _ = model(x)

        loss = criterion(output.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")

Epoch 1, Loss: 1547.44
Epoch 2, Loss: 901.86
Epoch 3, Loss: 744.12
Epoch 4, Loss: 684.49
Epoch 5, Loss: 656.38
Epoch 6, Loss: 640.91
Epoch 7, Loss: 631.39
Epoch 8, Loss: 624.44
Epoch 9, Loss: 619.77
Epoch 10, Loss: 615.16


In [6]:
def generate_name(model, max_len=20, temperature=1.1):
    model.eval()

    idx = stoi["^"]
    input = torch.tensor([[idx]])

    name = ""
    hidden = None

    for _ in range(max_len):
        output, hidden = model(input, hidden)

        logits = output[0, -1] / temperature
        probs = torch.softmax(logits, dim=0)

        idx = torch.multinomial(probs, 1).item()
        char = itos[idx]

        if char == "$":
            break

        name += char
        input = torch.tensor([[idx]])

    return name.capitalize()

In [7]:
print("Generated Names:\n")

for _ in range(20):
    print(generate_name(model))

Generated Names:

Jai jain
Tanvi jain
Shivani shetty
Isha randey
Dev kulkarnjay das
Chaitanya gill
Aditya shetty
Saani naidu
Charna tripathi
Zoya chauhan
Shivani banerjee
Raghav pillai
Bhavya gill
Amrita iyedhi
Aman namboodiri
Krishna iyer
Ayaan bose
Yogesh rastogi
Chaitanya kulkarni
Bhavya namboori


In [8]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters:", count_params(model))

Trainable parameters: 39963


In [9]:
def novelty_rate(generated, train_set):
    new_names = []

    for name in generated:
        name = name.strip().lower()
        if name not in train_set:
            new_names.append(name)

    return len(new_names) / len(generated)


def diversity(generated):
    return len(set(generated)) / len(generated)


generated = [generate_name(model, temperature=1.1) for _ in range(100)]

train_set = set([n[1:-1].strip().lower() for n in names])

print("Novelty:", novelty_rate(generated, train_set))
print("Diversity:", diversity(generated))

Novelty: 0.74
Diversity: 0.97


*  The vanilla RNN achieves a novelty score of 0.74, indicating that a majority of generated names are not present in the training set. The diversity score of 0.97 shows that the model produces highly varied outputs with minimal repetition.
*  The high novelty and diversity suggest that the model effectively learns character-level patterns and recombines them to generate new and diverse name sequences.
*  Despite high diversity, occasional unrealistic sequences are observed due to limitations of vanilla RNNs in modeling long-range dependencies.





## BLSTM

In [13]:
import torch.nn as nn

class CharBLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)

        self.lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)   # prevents overfitting

        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [14]:
hidden_size = 128
lr = 0.001
epochs = 5

blstm_model = CharBLSTM(vocab_size, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(blstm_model.parameters(), lr=lr)

for epoch in range(epochs):
    total_loss = 0

    for x, y in zip(X, Y):
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

        optimizer.zero_grad()
        output = blstm_model(x)

        loss = criterion(output.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"BLSTM Epoch {epoch+1}, Loss: {total_loss:.2f}")

BLSTM Epoch 1, Loss: 414.96
BLSTM Epoch 2, Loss: 10.22
BLSTM Epoch 3, Loss: 3.28
BLSTM Epoch 4, Loss: 1.99
BLSTM Epoch 5, Loss: 0.87


In [18]:
def generate_name_blstm(model, max_len=20, temperature=1.2):
    model.eval()

    idx = stoi["^"]
    input = torch.tensor([[idx]])

    name = ""

    for _ in range(max_len):
        output = model(input)   # no hidden state

        logits = output[0, -1] / temperature
        probs = torch.softmax(logits, dim=0)

        idx = torch.multinomial(probs, 1).item()
        char = itos[idx]

        if char == "$":
            break

        name += char
        input = torch.tensor([[idx]])

    return name.capitalize()

In [19]:
print("BLSTM Generated Names:\n")

for _ in range(20):
    print(generate_name_blstm(blstm_model))

BLSTM Generated Names:

Kuswdh sshooguni s
S
Zlnzogpannijumm ponv
S
V
Mmoowdde
Prjudde
Zjas
Pa wanivikcpojunupy 
J
Snor
V
Saj yygojur vittttip
V
Cczj

Zere
In
Um p b^kussh guus
Z bmul


* Despite achieving very low training loss, the BLSTM model generates incoherent and noisy sequences during inference
* This occurs due to the mismatch between bidirectional training (which has access to future context) and unidirectional generation (which does not), leading to poor generalization.
* The BLSTM model fails as a generative model despite strong training performance, highlighting the importance of causal modeling in sequence generation tasks.

## RNN with Basic Attention Mechanism

In [33]:
import torch.nn as nn
import torch

class CharRNNAttention(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)

        self.attn = nn.Linear(hidden_size, 1)

        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)   # (B, T, H)

        # attention scores
        scores = self.attn(out).squeeze(-1)   # (B, T)
        weights = torch.softmax(scores, dim=1)

        # context vector
        context = torch.bmm(weights.unsqueeze(1), out)  # (B,1,H)
        context = context.repeat(1, out.size(1), 1)     # (B,T,H)

        #  concatenate context + original
        out = torch.cat([out, context], dim=2)          # (B,T,2H)

        out = self.fc(out)

        return out

In [34]:
attn_model = CharRNNAttention(vocab_size, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(attn_model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    total_loss = 0

    for x, y in zip(X, Y):
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

        optimizer.zero_grad()

        output = attn_model(x)   # (B, T, vocab)

        loss = criterion(output.view(-1, vocab_size), y.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Attention Epoch {epoch+1}, Loss: {total_loss:.2f}")

Attention Epoch 1, Loss: 1401.61
Attention Epoch 2, Loss: 645.77
Attention Epoch 3, Loss: 415.05
Attention Epoch 4, Loss: 306.44
Attention Epoch 5, Loss: 240.55
Attention Epoch 6, Loss: 196.18
Attention Epoch 7, Loss: 165.26
Attention Epoch 8, Loss: 142.54
Attention Epoch 9, Loss: 129.48
Attention Epoch 10, Loss: 109.92


In [38]:
def generate_name_attn(model, max_len=20, temperature=1.5, top_k=5):
    model.eval()

    idx = stoi["^"]
    input = torch.tensor([[idx]])

    name = ""

    for _ in range(max_len):
        output = model(input)

        logits = output[0, -1] / temperature

        #  Top-K sampling
        values, indices = torch.topk(logits, top_k)

        probs = torch.softmax(values, dim=0)

        # sample from top-k
        sampled_idx = torch.multinomial(probs, 1).item()
        idx = indices[sampled_idx].item()

        char = itos[idx]

        if char == "$":
            break

        name += char
        input = torch.tensor([[idx]])

    return name.capitalize()

In [39]:
for _ in range(20):
    print(generate_name_attn(attn_model))

Clllllllllllllllllll
Pppppppppppppppppppt
Cxeeeeeeeeeeeeeeeeee
Llllllllllllllllllll
Zaxhhhhhannnnnnnnnnn
Clllllllllllllllllll
Llllllllllllllllllll
C^zwbbbbbbbbbbbbbbbb
Llllllllllllllllllll
Llllllllllllllllllll
C^zzwddddddddddddddd
Zzwavvvvvvvvvvvvvvvv
Ppppppppppiiiillllll
Zzzwbbbbbbbbbbbbbbbb
Jjjjjjjjjjjjjjjjjjjj
Jjjjjjjjjjjjjjjjjjjj
Pppppppppttttttttttt
Jjjjjjjjjjjjjjjjjjjj
Zannnnnnnnnnnnnnnnnn
Pppppppppppppppppppp


* The attention-based RNN showed unstable behavior during generation, often producing repetitive character sequences
* This is due to the model collapsing to high-probability characters, a common issue in small datasets and simple attention implementations.
* While attention theoretically improves sequence modeling, in this setting it requires more data and tuning to outperform vanilla RNN

Vanilla RNN performed most reliably for this dataset, while BLSTM failed due to non-causality, and attention required more tuning to stabilize.